# 📊 Thu thập chỉ số tài chính MỞ RỘNG — 46 mã, phục vụ điểm chất lượng tổng hợp
Lấy thêm từ bảng ratio VCI: **P/CF** (dòng tiền), **D/E** (đòn bẩy), **biên lợi nhuận**, **P/S** (suy ra tăng trưởng doanh thu).
Chỉ 46 mã nên chạy nhanh (~10−15 phút). Chạy trên Colab, cùng thư mục Drive `LuanVan2/DELAFO_taichinh`
(cần file `prepared_2018_2026.csv` để lấy danh sách mã).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/LuanVan2/DELAFO_taichinh

In [ ]:
%pip install -q vnstock
import pandas as pd, numpy as np, time, os

try:
    from vnstock.api.financial import Finance
    NEW_API = True
    print("✅ vnstock API mới (>=4.x)")
except ImportError:
    from vnstock import Vnstock
    NEW_API = False
    print("✅ vnstock API cũ")

def vs_ratio(ticker, src):
    if NEW_API:
        if src.upper() == "VCI":
            from vnstock.explorer.vci.financial import Finance as VciFinance
            f = VciFinance(symbol=ticker, period="quarter", show_log=False)
            return f._get_financial_report("ratio", period="quarter", lang="en", dropna=False, limit=200)
        return Finance(source=src.lower(), symbol=ticker, period="quarter").ratio(lang="en", dropna=False)
    return Vnstock().stock(symbol=ticker, source=src.upper()).finance.ratio(period="quarter", lang="en", dropna=False)

tickers = sorted(pd.read_csv("prepared_2018_2026.csv").ticker.unique())
print(f"{len(tickers)} mã:", " ".join(tickers))

In [ ]:
# ── Thu thập (có checkpoint — chạy lại được nếu đứt) ──
import re
DELAY = 4
CKPT = "ckpt_ratio_morong.csv"

KEYS = [("pe_ratio","P/E"), ("pb_ratio","P/B"), ("roe","ROEA_check"),
        ("price_to_cash_flow","P/CF"), ("debt_to_equity","D/E"),
        ("net_margin","NM"), ("gross_margin","GM"), ("ps_ratio","P/S")]

def parse_wide(df, ticker):
    per_cols = [c for c in df.columns if re.match(r"^\d{4}-Q[1-4]$", str(c))]
    if not per_cols or "item_id" not in df.columns:
        return None
    key = df["item_id"].astype(str).str.lower()
    data, miss = {}, []
    for k, name in KEYS:
        hit = df[key == k]
        if hit.empty:
            data[name] = [np.nan]*len(per_cols); miss.append(k)
        else:
            data[name] = pd.to_numeric(hit.iloc[0][per_cols], errors="coerce").values
    out = pd.DataFrame(data)
    out["year"]    = [int(c[:4]) for c in per_cols]
    out["quarter"] = [int(c[-1]) for c in per_cols]
    out["ticker"]  = ticker
    if miss: print(f"  ⚠️ {ticker} thiếu item: {miss}")
    return out

frames, done = [], set()
if os.path.exists(CKPT):
    prev = pd.read_csv(CKPT); done = set(prev.ticker.unique()); frames.append(prev)
    print(f"🔄 checkpoint: {len(done)} mã đã có")
rem = [t for t in tickers if t not in done]
for i, t in enumerate(rem, 1):
    out = None
    for src in ["VCI", "KBS"]:
        try:
            df = vs_ratio(t, src)
            if df is None or len(df) == 0: continue
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [c[-1] if isinstance(c, tuple) else c for c in df.columns]
            out = parse_wide(df.reset_index(), t)
            if out is not None: break
        except Exception as e:
            print(f"  🔍 {t} {src}: {type(e).__name__}: {str(e)[:80]}")
    if out is not None: frames.append(out)
    else: print(f"  ❌ {t}: bỏ qua")
    if frames and (i % 10 == 0 or i == len(rem)):
        pd.concat(frames, ignore_index=True).to_csv(CKPT, index=False)
        print(f"[{i}/{len(rem)}] đã lưu checkpoint")
    time.sleep(DELAY)
print("✅ Xong thu thập")

In [ ]:
# ── QA + xuất file ──
r = pd.read_csv(CKPT)
r["ROEA_check"] = r["ROEA_check"] * 100.0   # tỉ lệ -> %
print(f"{r.ticker.nunique()} mã, {len(r)} dòng quý")
print("Số quý mỗi mã (trung vị):", r.groupby("ticker").size().median())
print("\nĐộ phủ từng cột (%):")
print((r[["P/CF","D/E","NM","GM","P/S"]].notna().mean()*100).round(1).to_string())
print("\nFPT 3 quý gần nhất:")
print(r[r.ticker=="FPT"].sort_values(["year","quarter"]).tail(3)[["year","quarter","P/CF","D/E","NM","GM","P/S","ROEA_check"]].to_string(index=False))
# doi chieu ROEA voi du lieu cu de chac khong lech don vi
old = pd.read_csv("prepared_2018_2026.csv")
old_fpt = old[old.ticker=="FPT"]["ROEA"].dropna().iloc[-1]
print(f"\nROEA FPT: mới={r[r.ticker=='FPT'].sort_values(['year','quarter'])['ROEA_check'].dropna().iloc[-1]:.2f} vs cũ={old_fpt:.2f} (phải cùng cỡ)")

r.to_csv("ratio_morong_46.csv", index=False)
print("\n✅ Đã lưu ratio_morong_46.csv — tải file này gửi Claude")